# Day 61：毕业设计开发 Day 3：核心多 Agent 主链路（MCP 工具 + 多轮记忆 + 人工审批）（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 9 周**：星辰智脑毕业设计全栈开发与上线  
> **GitLab 工单**：`STARMIND-061` ｜ **开发分支**：`feat/day61-capstone-multi-agent-hitl`  
> **配套 HTML5 课件**：`courseware/day61/index.html` ｜ **独立版本快照**：`versions/day61/`

---

## 🎙️ 老陈开场旁白

早啊各位！毕业设计 Day 3，今天把星辰智脑的'大脑'真正立起来——多 Agent 编排 + MCP 工具协议 + 多轮会话记忆 + 人工审批（HITL）。我一直强调一个企业级红线：Agent 查资料可以全自动，但凡涉及'动真金白银'或者'改生产配置'（比如提交 5000 元报销单、重启线上服务），绝对不能让大模型一拍脑门直接执行！今天咱用纯 Python 写出一套清晰可单测的多 Agent 状态机：Supervisor 负责意图路由，KnowledgeAgent 调 RAG 工具，ActionAgent 调 MCP 业务工具，触发高危阈值自动挂起为 `PENDING_APPROVAL`，等主管点同意后再续跑！

### 承上启下
- **昨日基线**：昨天我们打通了星辰智脑的混合检索 RAG + Rerank + 引用溯源引擎，今天正式进入核心智能体主链路开发。
- **明日预告**：明天（Day 62）我们把 RAG 与多 Agent 引擎统一封装为 FastAPI 全量 REST/SSE 接口，并与前端控制台完成全链路联调。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day61'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-061）

星辰智脑不仅需要回答企业知识库问题，还要能代表员工执行请假、差旅报销提交、IT 工单派发等操作，并在多轮对话中记住上下文、对高风险操作强制引入人工审批。

**今日迭代目标：**
- 实现 MCP 风格工具注册中心：统一声明工具名、参数 Schema、风险等级与执行审计。
- 实现多轮会话记忆管理器：支持滑动窗口裁剪与历史摘要压缩，避免上下文超长。
- 实现 Supervisor 多 Agent 编排器与 HITL 审批状态机：低危自动执行，高危挂起待审批并支持批准/驳回恢复。

**验收标准：**
- 知识问答类请求准确路由至 `knowledge_agent`，业务办理类请求路由至 `action_agent`。
- 金额超过 2000 元的报销申请自动进入 `PENDING_APPROVAL`，批准后完成执行并写入审计轨迹，驳回则终止。
- 多轮记忆自动保留最近 N 轮对话并将更早历史压缩为摘要。

---

## 📚 课堂核心技术笔记

### 1. 为什么企业级 Agent 必须区分只读工具与副作用工具？

只读工具（如查知识库、查考勤余额）具备天然幂等性，可以放心由模型自主调用；而带外部副作用的写入工具（报销打款、删除工单、变更权限）一旦模型误判或被 Prompt 注入利用，后果不可逆，必须通过确定性代码层的风险策略函数拦截并交由人类审批。

> **⚠️ 老陈避坑**：切忌把'大于 2000 元请询问人工'只写在 System Prompt 里！安全门禁必须在 Python 代码层硬性拦截。

### 2. 多轮会话记忆的 Token 失控防护

随着对话轮数增加，全量拼接历史消息会导致延迟暴涨且费用线性攀升。采用'滚动摘要 + 最近 K 轮原文'的混合窗口策略，可在不丢关键实体的前提下把历史 Token 稳定封顶。

> **⚠️ 老陈避坑**：不要按字符硬截断 JSON 或代码块，应以完整 `turn`（user+assistant 对）为单位滑动裁剪。


## 💻 实操步骤 1：步骤一：实现 MCP 风格工具注册中心与多轮滑动窗口记忆

- **目标文件**：`versions/day61/src/day61_mcp_and_memory.py`
- **老陈代码旁白**：先搭好 Agent 赖以运转的两大底座：带参数校验与风险判定钩子的 MCP 工具注册表，以及带自动摘要折叠的多轮会话记忆。


In [ ]:
"""
Day 61 - 模块一：MCP 风格标准化工具注册中心 + 多轮会话滑动窗口记忆
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional


@dataclass
class MCPToolSpec:
    name: str
    description: str
    required_params: List[str]
    handler: Callable[[Dict[str, Any]], Dict[str, Any]]
    risk_checker: Callable[[Dict[str, Any]], bool] = lambda args: False


class MCPToolRegistry:
    """统一管理企业 MCP 工具的注册、参数校验、风险评估与审计执行"""

    def __init__(self) -> None:
        self._tools: Dict[str, MCPToolSpec] = {}
        self.audit_log: List[Dict[str, Any]] = []

    def register(self, spec: MCPToolSpec) -> None:
        self._tools[spec.name] = spec

    def list_tools(self) -> List[Dict[str, Any]]:
        return [
            {
                "name": t.name,
                "description": t.description,
                "required_params": list(t.required_params),
            }
            for t in self._tools.values()
        ]

    def validate_and_check_risk(self, name: str, args: Dict[str, Any]) -> bool:
        if name not in self._tools:
            raise KeyError(f"未注册的 MCP 工具: {name}")
        spec = self._tools[name]
        missing = [p for p in spec.required_params if p not in args or args[p] in (None, "")]
        if missing:
            raise ValueError(f"工具 {name} 缺少必填参数: {missing}")
        return bool(spec.risk_checker(args))

    def invoke(self, name: str, args: Dict[str, Any], operator: str = "agent") -> Dict[str, Any]:
        self.validate_and_check_risk(name, args)
        result = self._tools[name].handler(args)
        self.audit_log.append(
            {"tool": name, "args": dict(args), "operator": operator, "result": result}
        )
        return result


@dataclass
class MemorySession:
    session_id: str
    max_recent_messages: int = 4
    summary: str = ""
    messages: List[Dict[str, str]] = field(default_factory=list)


class SessionMemoryStore:
    """多轮对话记忆管理：保留最近窗口 + 超窗历史自动折叠为摘要"""

    def __init__(self, max_recent_messages: int = 4) -> None:
        self.max_recent_messages = max_recent_messages
        self._sessions: Dict[str, MemorySession] = {}

    def _get(self, session_id: str) -> MemorySession:
        if session_id not in self._sessions:
            self._sessions[session_id] = MemorySession(
                session_id=session_id,
                max_recent_messages=self.max_recent_messages,
            )
        return self._sessions[session_id]

    def add_turn(self, session_id: str, user_msg: str, assistant_msg: str) -> None:
        sess = self._get(session_id)
        sess.messages.append({"role": "user", "content": user_msg})
        sess.messages.append({"role": "assistant", "content": assistant_msg})
        if len(sess.messages) > sess.max_recent_messages:
            overflow = sess.messages[:-sess.max_recent_messages]
            sess.messages = sess.messages[-sess.max_recent_messages:]
            folded = " | ".join(f"{m['role']}:{m['content'][:30]}" for m in overflow)
            sess.summary = f"{sess.summary}; {folded}".strip("; ")

    def get_context(self, session_id: str) -> Dict[str, Any]:
        sess = self._get(session_id)
        return {
            "session_id": sess.session_id,
            "summary": sess.summary,
            "recent_messages": list(sess.messages),
        }


if __name__ == "__main__":
    reg = MCPToolRegistry()
    reg.register(
        MCPToolSpec(
            name="submit_expense",
            description="提交差旅费用报销单",
            required_params=["employee", "amount", "reason"],
            handler=lambda a: {"expense_id": "EXP-2026-001", "status": "SUBMITTED", "amount": a["amount"]},
            risk_checker=lambda a: float(a["amount"]) > 2000.0,
        )
    )
    print("工具列表:", reg.list_tools())
    print("3500元是否高危:", reg.validate_and_check_risk("submit_expense", {"employee": "小王", "amount": 3500, "reason": "上海出差"}))
    mem = SessionMemoryStore(max_recent_messages=2)
    mem.add_turn("s1", "查一下住宿标准", "一线城市650元/晚")
    mem.add_turn("s1", "帮我报销1200元", "已自动提交报销单")
    print("记忆上下文:", mem.get_context("s1"))


### 🔍 步骤 1 代码深度解析

`MCPToolRegistry` 将参数校验与高危判定收敛在工具执行入口之前，确保任何 Agent 节点都无法绕过；`SessionMemoryStore` 自动将超出窗口的早期消息折叠进 `summary`，兼顾上下文连贯性与成本可控。

**预期运行输出参考：**
```text
工具列表: [{'name': 'submit_expense', 'description': '提交差旅费用报销单', 'required_params': ['employee', 'amount', 'reason']}]
3500元是否高危: True
记忆上下文: {'session_id': 's1', 'summary': 'user:查一下住宿标准 | assistant:一线城市650元/晚', 'recent_messages': [...]}
```


## 💻 实操步骤 2：步骤二：实现 Supervisor 多 Agent 编排器与 HITL 人工审批状态机

- **目标文件**：`versions/day61/src/day61_multi_agent_hitl.py`
- **老陈代码旁白**：基于工具注册表与记忆层，实现 Supervisor 意图分发、KnowledgeAgent/ActionAgent 协作以及高危动作挂起/审批恢复闭环。


In [ ]:
"""
Day 61 - 模块二：星辰智脑 Supervisor 多 Agent 编排与 HITL 人工审批引擎
"""
from __future__ import annotations

import re
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from src.day61_mcp_and_memory import MCPToolRegistry, MCPToolSpec, SessionMemoryStore


@dataclass
class ApprovalTicket:
    ticket_id: str
    session_id: str
    tool_name: str
    tool_args: Dict[str, Any]
    status: str = "PENDING"  # PENDING | APPROVED_EXECUTED | REJECTED
    approver: str = ""
    decision_note: str = ""
    execution_result: Optional[Dict[str, Any]] = None


@dataclass
class AgentResponse:
    session_id: str
    routed_agent: str
    status: str  # COMPLETED | PENDING_APPROVAL | REJECTED
    reply: str
    ticket_id: Optional[str] = None
    trace: List[str] = field(default_factory=list)


class StarMindMultiAgent:
    """多 Agent 编排器：Supervisor -> KnowledgeAgent / ActionAgent + HITL 审批"""

    def __init__(self) -> None:
        self.registry = MCPToolRegistry()
        self.memory = SessionMemoryStore(max_recent_messages=4)
        self.tickets: Dict[str, ApprovalTicket] = {}
        self._register_default_tools()

    def _register_default_tools(self) -> None:
        self.registry.register(
            MCPToolSpec(
                name="rag_search",
                description="检索星辰智脑企业制度知识库",
                required_params=["query"],
                handler=self._mock_rag_tool,
                risk_checker=lambda a: False,
            )
        )
        self.registry.register(
            MCPToolSpec(
                name="submit_expense",
                description="提交员工费用报销单（超2000元需主管人工审批）",
                required_params=["employee", "amount", "reason"],
                handler=lambda a: {
                    "expense_id": f"EXP-{uuid.uuid4().hex[:6].upper()}",
                    "employee": a["employee"],
                    "amount": float(a["amount"]),
                    "reason": a["reason"],
                    "status": "PAID_QUEUED",
                },
                risk_checker=lambda a: float(a["amount"]) > 2000.0,
            )
        )

    @staticmethod
    def _mock_rag_tool(args: Dict[str, Any]) -> Dict[str, Any]:
        q = args["query"]
        if any(k in q for k in ("住宿", "差旅", "酒店")):
            return {"doc_id": "DOC-01", "answer": "一线城市差旅住宿上限为 650 元/晚，需附电子专票。[DOC-01]"}
        if "4291" in q or "熔断" in q:
            return {"doc_id": "DOC-02", "answer": "ERR_4291 为部门日 Token 配额硬熔断，请联系管理员提额。[DOC-02]"}
        return {"doc_id": "NONE", "answer": "知识库未命中明确条款，建议联系人工客服确认。"}

    def _route(self, user_input: str) -> str:
        if any(k in user_input for k in ("报销", "提交", "申请")):
            if re.search(r"\d+", user_input):
                return "action_agent"
        return "knowledge_agent"

    def chat(self, session_id: str, user_input: str, employee: str = "张明") -> AgentResponse:
        trace = ["supervisor:received"]
        agent = self._route(user_input)
        trace.append(f"supervisor:routed->{agent}")

        if agent == "knowledge_agent":
            res = self.registry.invoke("rag_search", {"query": user_input}, operator="knowledge_agent")
            trace.append(f"knowledge_agent:rag_search->{res['doc_id']}")
            reply = res["answer"]
            self.memory.add_turn(session_id, user_input, reply)
            return AgentResponse(session_id, agent, "COMPLETED", reply, trace=trace)

        # ActionAgent 解析金额并走风险门禁
        m = re.search(r"(\d+(?:\.\d+)?)", user_input)
        amount = float(m.group(1)) if m else 0.0
        tool_args = {"employee": employee, "amount": amount, "reason": user_input}
        high_risk = self.registry.validate_and_check_risk("submit_expense", tool_args)

        if high_risk:
            tid = f"APR-{uuid.uuid4().hex[:6].upper()}"
            self.tickets[tid] = ApprovalTicket(
                ticket_id=tid,
                session_id=session_id,
                tool_name="submit_expense",
                tool_args=tool_args,
            )
            trace.append(f"action_agent:hitl_paused->{tid}")
            reply = f"报销金额 {amount:.2f} 元超过自动审批上限(2000元)，已挂起等待主管审批，审批单号：{tid}。"
            self.memory.add_turn(session_id, user_input, reply)
            return AgentResponse(session_id, agent, "PENDING_APPROVAL", reply, ticket_id=tid, trace=trace)

        exec_res = self.registry.invoke("submit_expense", tool_args, operator="action_agent")
        trace.append(f"action_agent:auto_executed->{exec_res['expense_id']}")
        reply = f"报销单已自动提交成功！单号：{exec_res['expense_id']}，金额：{amount:.2f} 元。"
        self.memory.add_turn(session_id, user_input, reply)
        return AgentResponse(session_id, agent, "COMPLETED", reply, trace=trace)

    def resolve_approval(self, ticket_id: str, approve: bool, approver: str, note: str = "") -> ApprovalTicket:
        if ticket_id not in self.tickets:
            raise KeyError(f"审批单不存在: {ticket_id}")
        ticket = self.tickets[ticket_id]
        if ticket.status != "PENDING":
            raise RuntimeError(f"审批单 {ticket_id} 已处于终态 {ticket.status}，禁止重复审批")

        ticket.approver = approver
        ticket.decision_note = note
        if not approve:
            ticket.status = "REJECTED"
            self.memory.add_turn(ticket.session_id, f"[审批通知]{ticket_id}", f"主管 {approver} 已驳回：{note}")
            return ticket

        ticket.execution_result = self.registry.invoke(
            ticket.tool_name, ticket.tool_args, operator=f"hitl_approved_by:{approver}"
        )
        ticket.status = "APPROVED_EXECUTED"
        self.memory.add_turn(
            ticket.session_id,
            f"[审批通知]{ticket_id}",
            f"主管 {approver} 已批准并完成执行：{ticket.execution_result['expense_id']}",
        )
        return ticket


if __name__ == "__main__":
    engine = StarMindMultiAgent()
    r1 = engine.chat("sess-01", "请问北京出差住宿报销标准是多少？")
    r2 = engine.chat("sess-01", "帮我提交差旅报销 800 元")
    r3 = engine.chat("sess-01", "帮我提交服务器采购报销 4500 元")
    print("知识问答:", r1.routed_agent, r1.status, r1.reply)
    print("低危自动:", r2.routed_agent, r2.status, r2.reply)
    print("高危挂起:", r3.routed_agent, r3.status, r3.ticket_id)
    if r3.ticket_id:
        done = engine.resolve_approval(r3.ticket_id, approve=True, approver="林总监", note="同意采购")
        print("审批恢复:", done.status, done.execution_result)


### 🔍 步骤 2 代码深度解析

`StarMindMultiAgent` 将意图路由、只读检索、写操作风险拦截、审批单幂等终态校验与审计日志全部贯通，杜绝重复审批与越权执行。

**预期运行输出参考：**
```text
知识问答: knowledge_agent COMPLETED 一线城市差旅住宿上限为 650 元/晚，需附电子专票。[DOC-01]
低危自动: action_agent COMPLETED 报销单已自动提交成功！...
高危挂起: action_agent PENDING_APPROVAL APR-...
审批恢复: APPROVED_EXECUTED {'expense_id': 'EXP-...', ...}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day61/tests/test_day61.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 61 单元测试：多 Agent 路由、MCP 工具审计、多轮记忆与 HITL 审批"""
import sys
from pathlib import Path
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day61_multi_agent_hitl import StarMindMultiAgent


def test_knowledge_agent_routing():
    app = StarMindMultiAgent()
    res = app.chat("s1", "出差住宿标准是多少？")
    assert res.routed_agent == "knowledge_agent"
    assert res.status == "COMPLETED"
    assert "[DOC-01]" in res.reply


def test_low_risk_action_auto_executes():
    app = StarMindMultiAgent()
    res = app.chat("s2", "帮我报销打车费 350 元")
    assert res.routed_agent == "action_agent"
    assert res.status == "COMPLETED"
    assert len(app.registry.audit_log) == 1


def test_high_risk_action_hitl_approve_and_idempotency():
    app = StarMindMultiAgent()
    res = app.chat("s3", "帮我提交差旅报销 3800 元")
    assert res.status == "PENDING_APPROVAL"
    assert res.ticket_id is not None
    # 挂起期间工具尚未真正执行
    assert len(app.registry.audit_log) == 0

    done = app.resolve_approval(res.ticket_id, approve=True, approver="陈经理", note="核对发票无误")
    assert done.status == "APPROVED_EXECUTED"
    assert done.execution_result is not None
    assert len(app.registry.audit_log) == 1

    # 二次审批必须拦截，防止重复打款
    with pytest.raises(RuntimeError):
        app.resolve_approval(res.ticket_id, approve=True, approver="陈经理")


def test_high_risk_action_reject_and_memory_window():
    app = StarMindMultiAgent()
    app.chat("s4", "住宿标准")
    app.chat("s4", "报错 4291 怎么办")
    res = app.chat("s4", "申请报销 5000 元")
    rejected = app.resolve_approval(res.ticket_id, approve=False, approver="陈经理", note="缺少发票")
    assert rejected.status == "REJECTED"
    ctx = app.memory.get_context("s4")
    assert len(ctx["recent_messages"]) == 4
    assert ctx["summary"] != ""

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为 MCPToolRegistry 增加按风险等级查询工具列表

**题目要求：**
编写函数 `list_high_risk_candidates(registry, sample_args_map)` 返回给定测试参数下会触发高危审批的工具名称列表。

**思路提示：** 遍历 `sample_args_map.items()`，调用 `registry.validate_and_check_risk(name, args)` 过滤即可。


In [ ]:
from src.day61_multi_agent_hitl import StarMindMultiAgent

app = StarMindMultiAgent()
samples = {
    "rag_search": {"query": "住宿"},
    "submit_expense": {"employee": "小李", "amount": 5000, "reason": "采购"},
}
high_risk = [
    name for name, args in samples.items()
    if app.registry.validate_and_check_risk(name, args)
]
assert high_risk == ["submit_expense"]
print("高危工具识别通过:", high_risk)


**答案解析与评分标准：**

静态与动态风险扫描帮助运维在上线前梳理所有需要接入 HITL 审批的敏感工具。


### 作业 2（企业实战）：企业实战：给审批单增加超时自动过期机制（TTL）

**题目要求：**
编写 `is_ticket_expired(created_ts, now_ts, ttl_seconds=86400)`，当审批单超过 24 小时未处理时返回 `True` 禁止再执行。

**思路提示：** 判断 `(now_ts - created_ts) > ttl_seconds`。


In [ ]:
def is_ticket_expired(created_ts: float, now_ts: float, ttl_seconds: float = 86400.0) -> bool:
    return (now_ts - created_ts) > ttl_seconds


assert is_ticket_expired(1000.0, 90000.0) is True
assert is_ticket_expired(1000.0, 2000.0) is False
print("审批单 TTL 过期检查通过")


**答案解析与评分标准：**

生产中挂起超过 24 小时的业务审批单可能已经物是人非，自动过期可防止陈旧指令被误批准执行。


### 作业 3（架构进阶）：架构进阶：实现多级审批链（2000~10000 元经理批，>10000 元总监批）

**题目要求：**
实现 `required_approvers(amount: float) -> list[str]`，按金额分级返回所需审批角色列表。

**思路提示：** 分段判断：`<=2000` 返回 `[]`，`<=10000` 返回 `['dept_manager']`，否则返回 `['dept_manager', 'vp_finance']`。


In [ ]:
def required_approvers(amount: float) -> list[str]:
    if amount <= 2000:
        return []
    if amount <= 10000:
        return ["dept_manager"]
    return ["dept_manager", "vp_finance"]


assert required_approvers(1500) == []
assert required_approvers(6000) == ["dept_manager"]
assert required_approvers(25000) == ["dept_manager", "vp_finance"]
print("多级审批链路由通过")


**答案解析与评分标准：**

真实企业 OA 审批往往是串行或会签的多级审批链，通过规则表驱动即可平滑扩展。


---

## 🎯 老陈收尾复盘

漂亮！今天这套多 Agent + MCP + 记忆 + HITL 审批闭环，既有架构高度又极其干净。尤其是高危动作挂起、拿到审批单号后再恢复执行的设计，面试官一听就知道你做过真实企业落地。明天咱们用 FastAPI 把全量接口和 SSE 流式输出包出来！
